# IBGE Municipality Population - Bronze Ingestion

## Parameters

In [0]:
dbutils.widgets.text(name="environment", defaultValue="dev", label="Environment")

environment = dbutils.widgets.get("environment").strip().lower()

if environment not in ("dev", "test", "prod"):
    raise ValueError(
        f"Unsupported environment: {environment}. Expected dev, test, or prod."
    )

## Setup

In [0]:
import requests
from datetime import datetime
from pyspark.sql.functions import col, current_timestamp, lit, explode, year
import uuid
import time

In [0]:
catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "ibge_municipality_population"
target_table = f"{catalog}.{schema}.{table_name}"
source_system = "ibge"
source_dataset = "municipality_population"

run_id = str(uuid.uuid4())

aggregate_id = "6579"
variable_id = "9324"
municipality_level = "N6"
api_base_url = "https://servicodados.ibge.gov.br/api/v3/agregados"

politeness = 1
request_batch = 100
timeout=5

In [0]:
orders_table = f"{catalog}.silver.olist_orders"

required_years_df = (
    spark.table(orders_table)
    .select(year("order_purchase_timestamp").alias("year"))
    .where(col("year").isNotNull())
    .distinct()
    .orderBy("year")
)

required_years = [row["year"] for row in required_years_df.collect()]

In [0]:
if spark.catalog.tableExists(target_table):
    existing_years_df = (
        spark.table(target_table)
        .select(col("year").cast("int").alias("year"))
        .where(col("year").isNotNull())
        .distinct()
        .orderBy("year")
    )

    existing_years = [row["year"] for row in existing_years_df.collect()]
else:
    existing_years = []

In [0]:
missing_years = sorted(set(required_years) - set(existing_years))

In [0]:
if not missing_years:
    dbutils.notebook.exit("No missing population years to ingest.")

In [0]:
municipalities_df = spark.table(f"{catalog}.bronze.ibge_municipalities")
municipality_ids = municipalities_df.select("ibge_municipality_id").collect()
municipality_ids = [str(row[0]) for row in municipality_ids]

## Call the IBGE API in batches and save response to volume

In [0]:
response_file_paths = []
volume_path = "/Volumes/ecommerce_{environment}/landing/raw_files/api/{source_dataset}/{response_date}/{table_name}_api_response_{response_date_time}.json"

In [0]:
for population_year in missing_years:
    for i in range(0, len(municipality_ids), request_batch):
        municipality_ids_batch = municipality_ids[i:i + request_batch]

        api_url = (
            f"{api_base_url}/{aggregate_id}"
            f"/periodos/{population_year}"
            f"/variaveis/{variable_id}"
        )

        params = {
            "localidades": f"{municipality_level}[{','.join(municipality_ids_batch)}]"
        }

        response = requests.get(
            api_url,
            params=params,
            timeout=timeout
        )
        
        if response.status_code != 200:
            raise RuntimeError(f"Expected 200, got {response.status_code}")
        
        response_date = datetime.now().strftime("%Y%m%d")
        response_date_time = datetime.now().strftime("%Y%m%d_%H%M%S")

        write_file_path = volume_path.format(
            environment=environment,
            source_dataset=source_dataset,
            response_date=response_date,
            table_name=table_name,
            response_date_time=response_date_time
        )

        try:
            with open(write_file_path, "w", encoding="utf-8") as f:
                f.write(response.text)

        except FileNotFoundError:
            dbutils.fs.mkdirs("/Volumes/ecommerce_{environment}/landing/raw_files/api/{source_dataset}/{response_date}/".format(
                environment=environment,
                source_dataset=source_dataset,
                response_date=response_date)
            )

            with open(write_file_path, "w", encoding="utf-8") as f:
                f.write(response.text)
        
        response_file_paths.append(write_file_path)
        time.sleep(politeness)


## Read from volume

In [0]:
source_df = spark.read.json(response_file_paths)

In [0]:
# Explode the top level 'resultados' array
df_res = source_df.withColumn("resultados", explode("resultados"))

In [0]:
# Explode the inner 'series' array
df_series = df_res.withColumn("series", explode("resultados.series"))

In [0]:
year_columns = [str(population_year) for population_year in missing_years]

serie_df = df_series.select(
    col("series.localidade.id").alias("ibge_municipality_id"),
    col("series.localidade.nome").alias("municipality_name"),
    *[
        col(f"series.serie.`{population_year}`").alias(str(population_year))
        for population_year in missing_years
    ],
    col("_metadata.file_path").alias("source_file_path"),
    col("_metadata.file_modification_time").alias(
        "source_file_modification_time"
    )
)

In [0]:
bronze_df = (
    serie_df
    .unpivot(
        ids=[
            "ibge_municipality_id",
            "municipality_name",
            "source_file_path",
            "source_file_modification_time"
        ],
        values=year_columns,
        variableColumnName="year",
        valueColumnName="population"
    )
    .where(col("population").isNotNull())
)

## Add Bronze metadata

In [0]:
bronze_df = (
    bronze_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

## Write to Bronze

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(target_table)
)